<a href="https://colab.research.google.com/github/niksisons/neural_networks/blob/main/%D0%A0%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_%E2%84%967_%D0%98%D1%81%D0%BF%D0%BE%D0%BB%D1%8C%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D0%B5_%D0%B0%D0%B2%D1%82%D0%BE%D1%8D%D0%BD%D0%BA%D0%BE%D0%B4%D0%B5%D1%80%D0%B0_%D0%B4%D0%BB%D1%8F_%D1%80%D0%B5%D1%88%D0%B5%D0%BD%D0%B8%D1%8F_%D0%B7%D0%B0%D0%B4%D0%B0%D1%87_%D1%81%D0%B5%D0%BC%D0%B0%D0%BD%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%BE%D0%B9_%D1%81%D0%B5%D0%B3%D0%BC%D0%B5%D0%BD%D1%82%D0%B0%D1%86%D0%B8%D0%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Практическая работа №7. Использование автоэнкодера U-Net для решения задач семантической сегментации**


---

## **Задание №1. Обучение готовой модели семантической сегментации для классификации спутниковых снимков**


### **Введение**

Данная практическая работа посвящена применению глубокого обучения для семантической сегментации спутниковых снимков. Вы освоите обработку аэрофотоснимков с помощью нейронной сети архитектуры U-Net для выделения различных типов земной поверхности: зданий, дорог, растительности и других элементов ландшафта  .

В ходе работы будут рассмотрены:

- Основные принципы семантической сегментации изображений  
- Архитектура нейронной сети U-Net и её особенности   
- Подготовка и предобработка геопространственных данных с использованием `tf.data`  
- Обучение модели семантической сегментации  
- Оценка качества с использованием специализированных метрик  
- Применение обученной модели для анализа полноразмерных снимков

---

### **Раздел 1. Теоретические основы**

#### **1.1. Семантическая сегментация в геоанализе**

Семантическая сегментация — процесс разделения изображения на семантически значимые области, где каждому пикселю присваивается метка класса В контексте геоанализа это позволяет автоматически выделять на спутниковых снимках здания, дороги, водные объекты, растительность и т.д.

#### **1.2. Архитектура U-Net**

U-Net — сверточная нейронная сеть, изначально разработанная для сегментации биомедицинских изображений, но успешно адаптированная для геоанализа  . Название происходит от U-образной формы архитектуры.

Основные компоненты:

1. **Энкодер (downsampling)** — последовательность свёрточных слоёв с max-pooling для извлечения признаков **Декодер (upsampling)** — восстановление пространственного разрешения  
3. **Skip-connections** — обходные соединения между уровнями энкодера и декодера для сохранения детализации

---

### **Раздел 2. Подготовка данных**

#### **2.1. Загрузка и первичное знакомство с данными**

Скачайте набор данных по ссылке и распакуйте в рабочую директорию:

**Ссылка на датасет:** https://landcover.ai.linuxpolska.com/#dataset (снимки и маски, Version 1)


---

#### **2.2. Установка необходимых библиотек**

In [ ]:
%%capture
!pip install -U -q segmentation-models

import os
os.kill(os.getpid(), 9)  # перезапуск среды после установки

После перезапуска импортируйте библиотеки:

In [ ]:
import os
import glob
import random
import numpy as np
import cv2

from PIL import Image
from matplotlib import pyplot as plt

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.models import load_model

os.environ['SM_FRAMEWORK'] = 'tf.keras'
import segmentation_models as sm
from segmentation_models.losses import DiceLoss

from sklearn.metrics import confusion_matrix, cohen_kappa_score
import seaborn as sns

---

#### **Совет: работа с TIFF-изображениями нативно в TensorFlow**

Если ваш датасет содержит изображения в формате TIFF, вы можете работать с ними напрямую через `tensorflow-io`:

```bash
pip install tensorflow-io
```

In [ ]:
import tensorflow as tf
import tensorflow_io as tfio

def load_tiff(file_path):
    img = tf.io.read_file(file_path)
    # Декодирует TIFF. Используйте 'index' для многостраничных TIFF.
    img = tfio.experimental.image.decode_tiff(img, index=0)
    return img

ds = tf.data.Dataset.list_files("path/to/images/*.tif")
ds = ds.map(load_tiff)

Или можете воспользоваться сторонними библиотеками для работы с tiff ([tifffile](https://pypi.org/project/tifffile/), [rasterio](https://pypi.org/project/rasterio/))

---

#### **2.3. Определение параметров эксперимента**

Задайте основные гиперпараметры и описание классов вашего датасета:

In [ ]:
# Основные параметры датасета и модели
ROOT_DIR = "landcover_ai" # путь до датасета
PATCH_SIZE = 256 # размер патча для подачи в сеть 
BATCH_SIZE = 16 # размер батча
EPOCHS = 30 # количество эпох
NUM_CLASSES = 5 # фон (0), здания (1), лес (2), вода (3), дороги (4)

# Описание классов в формате (rgb_цвет) для RGB масок
# Настройте под свой датасет, если маски представлены RGB изображениями
CLASS_COLORS = {
    0: (0, 0, 0),        # Фон
    1: (255, 0, 0),      # Здания
    2: (0, 255, 0),      # Лес/растительность
    3: (0, 0, 255),      # Вода
    4: (255, 255, 0)     # Дороги
}

CLASS_NAMES = ["Background", "Building", "Woodland", "Water", "Road"]


---

#### **2.4. Сбор путей к изображениям и маскам**

Напишите функцию для сбора путей к изображениям и маскам из структуры вашего датасета. Работайте напрямую с исходной структурой, **без создания промежуточных папок с патчами**.

In [ ]:
def collect_file_paths(root_dir):
    """
    Собирает пути к изображениям и маскам.
    Адаптируйте под структуру вашего датасета.
    """
    image_paths = []
    mask_paths = []

    # Предполагается структура папок `images/` и `masks/` внутри ROOT_DIR
    images_dir = os.path.join(root_dir, 'images')
    masks_dir = os.path.join(root_dir, 'masks')

    if os.path.exists(images_dir) and os.path.exists(masks_dir):
        # Находим все изображения (учитываем форматы jpg, png, tif)
        for ext in ('*.tif', '*.tiff', '*.jpg', '*.png'):
            image_paths.extend(glob.glob(os.path.join(images_dir, ext)))

        # Для каждого изображения ищем соответствующую маску
        for img_path in image_paths:
            basename = os.path.basename(img_path)
            # Предполагаем, что имя маски совпадает с именем изображения
            # Для Landcover.ai маска может иметь суффикс _m.png или _mask.png, здесь ищем полное совпадение или типичные суффиксы
            mask_path = os.path.join(masks_dir, basename.replace('.jpg', '.png').replace('.tif', '_m.png')) # адаптируйте под ваши имена
            if os.path.exists(mask_path):
                mask_paths.append(mask_path)
            else:
                # Попробуем без суффикса или просто с расширением PNG/TIF
                alt_mask_path = os.path.join(masks_dir, basename)
                if os.path.exists(alt_mask_path):
                   mask_paths.append(alt_mask_path)

    # Убедимся, что списки сбалансированы
    assert len(image_paths) == len(mask_paths), "Количество изображений и масок не совпадает!"

    return sorted(image_paths), sorted(mask_paths)

all_image_paths, all_mask_paths = collect_file_paths(ROOT_DIR)

print(f"Найдено изображений: {len(all_image_paths)}")
print(f"Найдено масок: {len(all_mask_paths)}")


---

#### **2.5. Визуальный осмотр примеров**

Реализуйте функцию для визуализации случайных пар изображение–маска:

In [ ]:
def visualize_samples(image_paths, mask_paths, num_samples=3):
    """Визуализирует случайные пары изображение–маска."""
    indices = random.sample(range(len(image_paths)), min(num_samples, len(image_paths)))
    
    plt.figure(figsize=(15, 5 * num_samples))
    for i, idx in enumerate(indices):
        img_path = image_paths[idx]
        mask_path = mask_paths[idx]
        
        # Получаем данные изображений
        img = np.array(Image.open(img_path))
        mask = np.array(Image.open(mask_path))
        
        plt.subplot(num_samples, 2, 2 * i + 1)
        plt.title(f"Изображение {idx}")
        if img.shape[-1] == 3:  # RGB
             plt.imshow(img)
        else: # Градации серого или другие форматы, отображаем первую полосу
             plt.imshow(img[:, :, 0], cmap='gray')
        plt.axis("off")
        
        plt.subplot(num_samples, 2, 2 * i + 2)
        plt.title(f"Маска {idx}")
        # Если маска RGB, отображаем как есть, иначе через палитру cmap
        if len(mask.shape) == 3:
            plt.imshow(mask)
        else:
            plt.imshow(mask, cmap='jet')
        plt.axis("off")
        
    plt.tight_layout()
    plt.show()

visualize_samples(all_image_paths, all_mask_paths, num_samples=3)

---

#### **2.6. Разбиение на обучающую и валидационную выборки**

Реализуйте **стратифицированное разбиение** данных, чтобы обеспечить присутствие всех географических областей в обеих выборках:

In [ ]:
from sklearn.model_selection import train_test_split

# Стратифицированное разбиение может основываться на регионах, 
# если в именах файлов есть префиксы, или просто случайное разбиение.
# В данном случае выполним стандартное случайное разбиение 80/20.
train_img_paths, val_img_paths, train_mask_paths, val_mask_paths = train_test_split(
    all_image_paths, all_mask_paths, test_size=0.2, random_state=42
)

print(f"Обучающих изображений: {len(train_img_paths)}")
print(f"Валидационных изображений: {len(val_img_paths)}")

---

#### **2.7. Подсчёт ожидаемого количества патчей**

Поскольку разбиение на патчи выполняется «на лету», полезно заранее оценить количество патчей:

In [ ]:
def count_patches(image_paths, patch_size=256):
    """Подсчитывает общее количество патчей."""
    total = 0
    for img_path in image_paths:
        # Открываем изображение для получения его ширины и высоты без загрузки данных в память
        with Image.open(img_path) as img:
            w, h = img.size
            total += (w // patch_size) * (h // patch_size)
    return total

num_train_patches = count_patches(train_img_paths, PATCH_SIZE)
num_val_patches = count_patches(val_img_paths, PATCH_SIZE)

print(f"Обучающих патчей: {num_train_patches}")
print(f"Валидационных патчей: {num_val_patches}")

---

### **Раздел 3. Предобработка данных и формирование tf.data-пайплайна**

#### **3.1. Преобразование RGB-масок в индексы классов**

Если маски представлены как цветные изображения, реализуйте векторизованную TensorFlow-функцию для преобразования RGB → индексы классов:

In [ ]:
def rgb_to_index(mask):
    """Преобразует RGB-маску в одноканальную индексированную маску."""
    # Собираем тензор из цветов классов
    # CLASS_COLORS содержит словари {id: (r, g, b)}
    colors_tensor = tf.constant([CLASS_COLORS[i] for i in range(NUM_CLASSES)], dtype=tf.uint8)
    
    # Расширяем размерности для Broadcasting
    # mask: (H, W, 3) -> (H, W, 1, 3)
    mask = tf.expand_dims(mask, axis=-2)
    # colors_tensor: (N, 3) -> (1, 1, N, 3)
    colors_tensor = tf.reshape(colors_tensor, (1, 1, NUM_CLASSES, 3))
    
    # Сравниваем маску со всеми цветами (H, W, N)
    matches = tf.reduce_all(tf.equal(mask, colors_tensor), axis=-1)
    
    # Получаем индекс совпавшего цвета. Возвращает (H, W)
    index_mask = tf.argmax(matches, axis=-1, output_type=tf.int32)
    
    # Возвращаем размерность канала (H, W, 1)
    return tf.expand_dims(index_mask, axis=-1)

---

#### **3.2. Функции загрузки и проверки данных**

In [ ]:
def load_pair(img_path, mask_path):
    """Загружает изображение и маску из файлов."""
    # Чтение изображения
    img_raw = tf.io.read_file(img_path)
    
    # Пытаемся декодировать TIFF, если расширение tif
    # В противном случае декодируем как jpeg/png
    img = tf.cond(
        tf.strings.regex_full_match(img_path, ".*\\.tif(f)?$"),
        lambda: tfio.experimental.image.decode_tiff(img_raw, index=0) if 'tfio' in globals() else tf.image.decode_image(img_raw, channels=3),
        lambda: tf.image.decode_image(img_raw, channels=3)
    )
    img.set_shape([None, None, 3])
    
    # Чтение маски
    mask_raw = tf.io.read_file(mask_path)
    mask = tf.cond(
        tf.strings.regex_full_match(mask_path, ".*\\.tif(f)?$"),
        lambda: tfio.experimental.image.decode_tiff(mask_raw, index=0) if 'tfio' in globals() else tf.image.decode_image(mask_raw, channels=None),
        lambda: tf.image.decode_image(mask_raw, channels=None)
    )
    mask.set_shape([None, None, None])

    return img, mask


def check_shapes(img, mask):
    """Проверяет совпадение размеров изображения и маски."""
    tf.debugging.assert_equal(
        tf.shape(img)[:2], tf.shape(mask)[:2],
        message="Размеры изображения и маски не совпадают"
    )
    return img, mask

---

#### **3.3. Извлечение патчей с помощью tf.image.extract_patches**

Реализуйте функцию для извлечения патчей «на лету» без сохранения на диск:

In [ ]:
def extract_patches(image, mask):
    """Извлекает патчи размером PATCH_SIZE из оригинальных изображений и масок."""
    img_patches = tf.image.extract_patches(
        images=tf.expand_dims(image, 0),
        sizes=[1, PATCH_SIZE, PATCH_SIZE, 1],
        strides=[1, PATCH_SIZE, PATCH_SIZE, 1],
        rates=[1, 1, 1, 1],
        padding='VALID'
    )
    
    mask_patches = tf.image.extract_patches(
        images=tf.expand_dims(mask, 0),
        sizes=[1, PATCH_SIZE, PATCH_SIZE, 1],
        strides=[1, PATCH_SIZE, PATCH_SIZE, 1],
        rates=[1, 1, 1, 1],
        padding='VALID'
    )
    
    # img_patches имеет размер (1, N_y, N_x, patch_size * patch_size * c)
    # Формируем их в тензор (N, patch_size, patch_size, c)
    img_patches = tf.reshape(img_patches, [-1, PATCH_SIZE, PATCH_SIZE, tf.shape(image)[-1]])
    mask_patches = tf.reshape(mask_patches, [-1, PATCH_SIZE, PATCH_SIZE, tf.shape(mask)[-1]])
    
    return tf.data.Dataset.from_tensor_slices((img_patches, mask_patches))

---

#### **3.4. Синхронная аугментация**

Реализуйте функцию аугментации, которая применяет одинаковые преобразования к изображению и маске:

In [ ]:
def augment_pair(img, mask):
    """Синхронная аугментация изображения и маски."""
    if tf.random.uniform(()) > 0.5:
        # Горизонтальное отражение
        img = tf.image.flip_left_right(img)
        mask = tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        # Вертикальное отражение
        img = tf.image.flip_up_down(img)
        mask = tf.image.flip_up_down(mask)
        
    return img, mask

---

#### **3.5. Предобработка изображений и масок**

ОБРАТИТЕ ВНИМАНИЕ НА ТО, В КАКОМ ФОРМАТЕ МАСКИ ИЗНАЧАЛЬНО ПРЕДСТАВЛЕНЫ, И В КАКОЙ ПАЛИТРЕ!

In [ ]:
def preprocess_pair(img, mask):
    """Предобработка изображения и маски перед подачей в пайплайн."""
    # Нормируем изображение [0, 1] или согласно предобученному энкодеру
    img = tf.cast(img, tf.float32) / 255.0
    
    # Преобразование маски в индексы классов
    if tf.shape(mask)[-1] == 3:
        mask = rgb_to_index(mask)
    else:
        mask = tf.cast(mask, tf.int32)
        
    # Преобразуем в One-Hot Encoding формат (требование segmentation_models)
    mask = tf.squeeze(mask, axis=-1) # удаляем лишнюю размерность, если она есть
    mask = tf.one_hot(mask, depth=NUM_CLASSES)
    
    return img, mask

---

#### **3.6. Создание оптимизированного tf.data-пайплайна**

Объедините все шаги в единую функцию создания датасета:

In [ ]:
def build_pipeline(image_paths, mask_paths, batch_size, shuffle=True, augment=False):
    """Создает оптимизированный tf.data пайплайн."""
    # 1. Создаем набор из пар путей
    ds = tf.data.Dataset.from_tensor_slices((image_paths, mask_paths))
    
    if shuffle:
        ds = ds.shuffle(buffer_size=len(image_paths), reshuffle_each_iteration=True)
    
    # 2. Загрузка пар изображений и масок
    ds = ds.map(load_pair, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(check_shapes, num_parallel_calls=tf.data.AUTOTUNE)
    
    # 3. Извлечение патчей
    # .interleave применяет функцию к каждому элементу и "раскручивает" возвращенные датасеты
    ds = ds.interleave(extract_patches, cycle_length=tf.data.AUTOTUNE, block_length=16)
    
    # 4. Предобработка и аугментация
    ds = ds.map(preprocess_pair, num_parallel_calls=tf.data.AUTOTUNE)
    
    if augment:
        ds = ds.map(augment_pair, num_parallel_calls=tf.data.AUTOTUNE)
    
    # 5. Батчинг и предзагрузка
    ds = ds.batch(batch_size, drop_remainder=True)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    
    return ds

train_ds = build_pipeline(train_img_paths, train_mask_paths, BATCH_SIZE, shuffle=True, augment=True)
val_ds = build_pipeline(val_img_paths, val_mask_paths, BATCH_SIZE, shuffle=False, augment=False)

---

#### **3.7. Проверка содержимого датасетов**

Реализуйте функцию для проверки и визуализации примеров из датасета:

In [ ]:
def check_dataset(ds, name, num_samples=3):
    """Проверяет датасет: размерности, типы, визуализация."""
    print(f"--- Проверка: {name} ---")
    
    # Берем один батч из датасета
    for img_batch, mask_batch in ds.take(1):
        print(f"Размер батча изображений: {img_batch.shape}, Тип данных: {img_batch.dtype}")
        print(f"Размер батча масок: {mask_batch.shape}, Тип данных: {mask_batch.dtype}")
        
        # Уникальные метки в масках (должны быть 0..4)
        unique_labels = np.unique(mask_batch.numpy())
        print(f"Уникальные метки классов в батче: {unique_labels}")
        
        plt.figure(figsize=(12, 4 * num_samples))
        
        for i in range(num_samples):
            # Отображение
            img = img_batch[i].numpy()
            mask = mask_batch[i].numpy()
            if mask.shape[-1] == 1:
                mask = np.squeeze(mask, axis=-1)
                
            plt.subplot(num_samples, 2, i*2 + 1)
            plt.imshow(img)
            plt.title(f"Image {i+1} ({img.shape})")
            plt.axis("off")
            
            plt.subplot(num_samples, 2, i*2 + 2)
            plt.imshow(mask, cmap='jet')
            plt.title(f"Mask {i+1} ({mask.shape})")
            plt.axis("off")
            
        plt.tight_layout()
        plt.show()

check_dataset(train_ds, "Обучающая выборка", num_samples=3)
check_dataset(val_ds, "Валидационная выборка", num_samples=2)

---

### **Раздел 4. Построение архитектуры и обучение модели**

#### **4.1. Определение метрик**

Используйте **несколько** метрик, доступных в библиотеке [Segmentation Models](https://segmentation-models.readthedocs.io/en/latest/api.html#metrics) для оценки качества обучения модели

In [ ]:
# Определение метрик из встроенных в segmentation_models
metrics = [
    sm.metrics.IOUScore(threshold=0.5, name='iou_score'),
    sm.metrics.FScore(threshold=0.5, name='f1_score')
]

---

#### **4.2. Инициализация модели U-Net**

Используйте библиотеку `segmentation_models` для создания U-Net с предобученным энкодером:

In [ ]:
BACKBONE = 'resnet34'
preprocess_input = sm.get_preprocessing(BACKBONE)

# Используем модель U-Net с энкодером resnet34 и весами, предобученными на ImageNet
model = sm.Unet(
    BACKBONE,
    classes=NUM_CLASSES,
    activation='softmax',        # softmax для многоклассовой классификации
    encoder_weights='imagenet',
)

model.summary()

Изучите [документацию Segmentation Models](https://segmentation-models.readthedocs.io/en/latest/tutorial.html) для выбора различных:
- [Функций потерь](https://segmentation-models.readthedocs.io/en/latest/api.html#losses)
- [Метрик](https://segmentation-models.readthedocs.io/en/latest/api.html#metrics)


---

#### **4.3. Компиляция модели**

In [ ]:
# Комбинированная функция потерь (Dice Loss + Categorical Crossentropy)
# отлично подходит для задач семантической сегментации при дисбалансе классов
dice_loss = sm.losses.DiceLoss() 
cce_loss = sm.losses.CategoricalCELoss()
total_loss = dice_loss + (1 * cce_loss)

# Компиляция модели
model.compile(
    optimizer=Adam(learning_rate=0.0001), 
    loss=total_loss, 
    metrics=metrics
)

---

#### **4.4. Обучение модели с колбэками**

In [ ]:
callbacks = [
    # Сохраняем модель с лучшим IoU на валидации
    ModelCheckpoint(
        filepath='best_unet_model.h5',
        monitor='val_iou_score', 
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    # Останавливаем обучение при отсутствии прогресса в течение 7 эпох
    EarlyStopping(
        monitor='val_iou_score',
        patience=7,
        mode='max',
        restore_best_weights=True,
        verbose=1
    )
]

print("Начинаем обучение модели...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

---

### **Раздел 5. Анализ результатов и визуализация**

#### **5.1. Визуализация процесса обучения**

Постройте графики изменения функции потерь и метрики IoU на обучении и валидации:

In [ ]:
plt.figure(figsize=(15, 5))

# График потерь (Loss)
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Обучение (Loss)')
plt.plot(history.history['val_loss'], label='Валидация (Loss)')
plt.title('Изменение функции потерь')
plt.xlabel('Эпохи')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# График метрики (IoU)
plt.subplot(1, 2, 2)
plt.plot(history.history['iou_score'], label='Обучение (IoU)')
plt.plot(history.history['val_iou_score'], label='Валидация (IoU)')
plt.title('Изменение метрики IoU')
plt.xlabel('Эпохи')
plt.ylabel('IoU Score')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

---

#### **5.2. Визуализация предсказаний на валидационных патчах**

In [ ]:
def visualize_predictions(dataset, model, num_samples=3):
    """Визуализация предсказаний на одном батче из датасета."""
    for img_batch, mask_batch in dataset.take(1):
        
        # Получаем предсказание модели
        preds = model.predict(img_batch)
        
        # Преобразуем (B, H, W, CLASSES) -> (B, H, W), берем индекс с наибольшей вероятностью
        preds_argmax = np.argmax(preds, axis=-1)
        
        # Аналогично возвращаем маску из One-Hot в (B, H, W)
        mask_argmax = np.argmax(mask_batch.numpy(), axis=-1)
        
        plt.figure(figsize=(15, 5 * num_samples))
        for i in range(num_samples):
            # Оригинал
            plt.subplot(num_samples, 3, i * 3 + 1)
            plt.imshow(img_batch[i])
            plt.title("Оригинальное изображение")
            plt.axis("off")
            
            # Маска (Истина)
            plt.subplot(num_samples, 3, i * 3 + 2)
            plt.imshow(mask_argmax[i], cmap='jet', vmin=0, vmax=NUM_CLASSES-1)
            plt.title("Истинная маска")
            plt.axis("off")
            
            # Предсказание
            plt.subplot(num_samples, 3, i * 3 + 3)
            plt.imshow(preds_argmax[i], cmap='jet', vmin=0, vmax=NUM_CLASSES-1)
            plt.title("Предсказание модели")
            plt.axis("off")
            
        plt.tight_layout()
        plt.show()

# Визуализация 3 примеров из валидационного датасета
visualize_predictions(val_ds, model, num_samples=3)

---

#### **5.3. Расчёт метрик по классам**

Вычислите IoU по каждому классу, средний IoU и коэффициент каппа Коэна:

In [ ]:
from sklearn.metrics import confusion_matrix, cohen_kappa_score
import seaborn as sns

# Сбор всех предсказаний и истинных меток для расчета метрик
# Так как батчей может быть много, сделаем это для нескольких (например, для всего val_ds)
y_true_all = []
y_pred_all = []

print("Идет получение предсказаний на валидационной выборке...")

for img_batch, mask_batch in val_ds:
    # Предсказания модели
    preds = model.predict(img_batch, verbose=0)
    
    # Преобразование вероятностей и one-hot к индексам классов (B, H, W) и уплощение массивов в 1D (число элементов: B*H*W)
    y_pred_all.extend(np.argmax(preds, axis=-1).flatten())
    y_true_all.extend(np.argmax(mask_batch.numpy(), axis=-1).flatten())

y_true_all = np.array(y_true_all)
y_pred_all = np.array(y_pred_all)

# Построение матрицы ошибок
# Строки - истинные классы, столбцы - предсказанные
cm = confusion_matrix(y_true_all, y_pred_all, labels=list(range(NUM_CLASSES)))

# IoU по классам = Диагональ (True Positives) / (Колонка (Preds) + Строка (Reals) - Диагональ (TP))
# То есть IoU = Intersection / Union
intersection = np.diag(cm)
ground_truth_set = cm.sum(axis=1) # строки
predicted_set = cm.sum(axis=0)    # столбцы
union = ground_truth_set + predicted_set - intersection

iou_per_class = intersection / (union + 1e-6) # избегаем деления на ноль

print("--- IoU по классам ---")
for i, name in enumerate(CLASS_NAMES):
    print(f"{name}: {iou_per_class[i]:.4f}")

mean_iou = np.mean(iou_per_class)
print(f"\nMean IoU: {mean_iou:.4f}")

# Рассчет Kappa-коэффициента Коэна (работает долго на больших массивах, можно сделать на срезе при необходимости)
# kappa = cohen_kappa_score(y_true_all, y_pred_all) 
# print(f"Cohen's Kappa: {kappa:.4f}")

# Тепловая карта матрицы ошибок
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Матрица ошибок (Confusion Matrix)')
plt.xlabel('Предсказанные классы (Predited)')
plt.ylabel('Истинные классы (Ground Truth)')
plt.show()

---

### **Раздел 6. Применение модели к полноразмерным снимкам**

#### **6.1. Быстрая сегментация полноразмерного изображения**

Реализуйте функцию для сегментации больших изображений путём разбиения на патчи и батчевого предсказания:

In [ ]:
def fast_predict(image, model, patch_size=256):
    """
    Сегментация большого изображения путем нарезки на патчи без перекрытия
    """
    h, w, c = image.shape
    
    # Дополняем изображение до размеров, кратных patch_size
    pad_h = (patch_size - h % patch_size) % patch_size
    pad_w = (patch_size - w % patch_size) % patch_size
    
    padded_img = np.pad(image, ((0, pad_h), (0, pad_w), (0, 0)), mode='reflect')
    padded_h, padded_w, _ = padded_img.shape
    
    mask_full = np.zeros((padded_h, padded_w, NUM_CLASSES), dtype=np.float32)
    
    # Нарезаем, предсказываем и склеиваем
    for i in range(0, padded_h, patch_size):
        for j in range(0, padded_w, patch_size):
            patch = padded_img[i:i+patch_size, j:j+patch_size]
            # Предобработка как при обучении
            patch = patch.astype(np.float32) / 255.0 
            patch = np.expand_dims(patch, axis=0) # Добавляем batch dimension
            
            # Предсказание
            pred = model.predict(patch, verbose=0)
            
            # Вставляем обратно в полную маску
            mask_full[i:i+patch_size, j:j+patch_size] = pred[0]
            
    # Обрезаем паддинг и берем класс с максимальной вероятностью
    final_mask = mask_full[:h, :w, :]
    return np.argmax(final_mask, axis=-1)

---

#### **6.2. Сегментация с плавным смешиванием**

Для уменьшения артефактов на границах патчей реализуйте сегментацию с перекрывающимися патчами и гауссовым взвешиванием:

In [ ]:
def smooth_predict(image, model, patch_size=256, stride=128):
    """
    Сегментация с плавным смешиванием (перекрывающиеся патчи + гауссовы веса).
    Существенно снижает артефакты-сетку на границах склеивания.
    """
    import scipy.stats as st
    
    h, w, c = image.shape
    
    # Гауссова маска весов
    x = np.linspace(-2, 2, patch_size)
    y = np.linspace(-2, 2, patch_size)
    x, y = np.meshgrid(x, y)
    gaussian_kernel = np.exp(-(x**2 + y**2))
    gaussian_kernel = gaussian_kernel / gaussian_kernel.max()
    gaussian_kernel = np.expand_dims(gaussian_kernel, axis=-1)
    
    # Добавляем padding
    pad_h = (stride - h % stride) % stride
    pad_w = (stride - w % stride) % stride
    padded_img = np.pad(image, ((0, pad_h), (0, pad_w), (0, 0)), mode='reflect')
    padded_h, padded_w, _ = padded_img.shape
    
    # Хранилище для суммы вероятностей и суммы весов
    mask_sum = np.zeros((padded_h, padded_w, NUM_CLASSES), dtype=np.float32)
    weight_sum = np.zeros((padded_h, padded_w, 1), dtype=np.float32)
    
    # Скользящее окно с шагом stride
    for i in range(0, padded_h - patch_size + 1, stride):
        for j in range(0, padded_w - patch_size + 1, stride):
            patch = padded_img[i:i+patch_size, j:j+patch_size]
            patch = patch.astype(np.float32) / 255.0
            patch = np.expand_dims(patch, axis=0) # [1, 256, 256, 3]
            
            # Получаем вероятности предсказаний
            pred = model.predict(patch, verbose=0)[0] # Убираем dimension батча [256, 256, NUM_CLASSES]
            
            # Умножаем предсказание на Гауссово окно распределения
            pred_weighted = pred * gaussian_kernel
            
            # Суммируем
            mask_sum[i:i+patch_size, j:j+patch_size] += pred_weighted
            weight_sum[i:i+patch_size, j:j+patch_size] += gaussian_kernel
            
    # Нормализуем по весам и обрезаем паддинг
    final_prob = mask_sum / (weight_sum + 1e-6)
    final_prob = final_prob[:h, :w, :]
    
    # Выбираем максимальную вероятность
    return np.argmax(final_prob, axis=-1)

---

#### **6.3. Сравнение предсказанной и оригинальной масок**

Реализуйте функцию для визуального сравнения результатов с расчётом метрик для отдельного снимка:

In [ ]:
def evaluate_on_full_image(img_path, mask_path, model):
    """Визуальное сравнение fast_predict и smooth_predict на одном полном изображении."""
    # Загружаем полное изображение и маску
    img = np.array(Image.open(img_path))
    mask = np.array(Image.open(mask_path))
    
    # Преобразуем RGB-маску в одноканальную
    if len(mask.shape) == 3:
        # Упрощенное приведение RGB к индексам для NumPy
        colors = [CLASS_COLORS[i] for i in range(NUM_CLASSES)]
        mask_idx = np.zeros(mask.shape[:2], dtype=np.uint8)
        for i, c in enumerate(colors):
            mask_idx[(mask == c).all(axis=-1)] = i
        mask = mask_idx
    
    print(f"Размер полного изображения: {img.shape}")
    
    # Получаем предсказания
    print("Выполнение fast_predict...")
    pred_fast = fast_predict(img, model, patch_size=256)
    
    print("Выполнение smooth_predict...")
    pred_smooth = smooth_predict(img, model, patch_size=256, stride=128)
    
    # Рассчет IoU
    iou_fast = cohen_kappa_score(mask.flatten(), pred_fast.flatten())
    iou_smooth = cohen_kappa_score(mask.flatten(), pred_smooth.flatten())
    print(f"Cohen Kappa Fast: {iou_fast:.4f}")
    print(f"Cohen Kappa Smooth: {iou_smooth:.4f}")
    
    # Визуализация
    plt.figure(figsize=(24, 6))
    
    plt.subplot(1, 4, 1)
    plt.imshow(img)
    plt.title('Оригинал')
    plt.axis('off')
    
    plt.subplot(1, 4, 2)
    plt.imshow(mask, cmap='jet', vmin=0, vmax=NUM_CLASSES-1)
    plt.title('Истинная маска')
    plt.axis('off')
    
    plt.subplot(1, 4, 3)
    plt.imshow(pred_fast, cmap='jet', vmin=0, vmax=NUM_CLASSES-1)
    plt.title('Fast Predict (Сетка)')
    plt.axis('off')
    
    plt.subplot(1, 4, 4)
    plt.imshow(pred_smooth, cmap='jet', vmin=0, vmax=NUM_CLASSES-1)
    plt.title('Smooth Predict (Гауссиан)')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()

# Запускаем на первой картинке из датасета
evaluate_on_full_image(all_image_paths[0], all_mask_paths[0], model)

---

## **Задание №2. Создание и обучение модели семантической сегментации для классификации спутниковых снимков**

### **Пример создания модели с Unet-подобной архитектурой**

In [ ]:
from tensorflow.keras import Model, Input
from keras.layers import Conv2D, MaxPooling2D, concatenate, UpSampling2D, Dropout, Cropping2D, Softmax, Conv2DTranspose
from keras.preprocessing import image
from keras.utils.vis_utils import plot_model

In [ ]:
def mini_u_net(image_shape, num_of_classes):

  input_image = Input(image_shape)

  # Encoder

  conv1_1 = Conv2D(filters = 32, kernel_size = (3, 3), padding = 'same', activation = 'relu', name = 'conv1_1')(input_image)
  conv1_2 = Conv2D(filters = 32, kernel_size = (3, 3), padding = 'same', activation = 'relu', name = 'conv1_2')(conv1_1)

  pool_1 = MaxPooling2D(name = 'pool_1')(conv1_2)

  conv2_1 = Conv2D(filters = 64, kernel_size = (3, 3), padding = 'same', activation = 'relu', name = 'conv2_1')(pool_1)
  conv2_2 = Conv2D(filters = 64, kernel_size = (3, 3), padding = 'same', activation = 'relu', name = 'conv2_2')(conv2_1)

  pool_2 = MaxPooling2D(name = 'pool_2')(conv2_2)

  conv3_1 = Conv2D(filters = 128, kernel_size = (3, 3), padding = 'same', activation = 'relu', name = 'conv3_1')(pool_2)
  conv3_2 = Conv2D(filters = 128, kernel_size = (3, 3), padding = 'same', activation = 'relu', name = 'conv3_2')(conv3_1)

  pool_3 = MaxPooling2D(name = 'pool_3')(conv3_2)

  conv4_1 = Conv2D(filters = 256, kernel_size = (3, 3), padding = 'same', activation = 'relu', name = 'conv4_1')(pool_3)
  conv4_2 = Conv2D(filters = 256, kernel_size = (3, 3), padding = 'same', activation = 'relu', name = 'conv4_2')(conv4_1)



  # Decoder

  upconv5_1 = UpSampling2D(name = 'upconv5_1')(conv4_2)
  upconv5_2 = Conv2D(filters = 128, kernel_size = (2, 2), activation = 'relu', padding = 'same', name = 'upconv5_2')(upconv5_1)
  concat_5 = concatenate([upconv5_2, conv3_2], axis = 3, name = 'concat_5') # Split Connections

  conv5_1 = Conv2D(filters = 128, kernel_size = (3, 3), padding = 'same', activation = 'relu', name = 'conv5_1')(concat_5)
  conv5_2 = Conv2D(filters = 128, kernel_size = (3, 3), padding = 'same', activation = 'relu', name = 'conv5_2')(conv5_1)


  upconv6_1 = UpSampling2D(name = 'upconv6_1')(conv5_2)
  upconv6_2 = Conv2D(filters = 64, kernel_size = (2, 2), activation = 'relu', padding = 'same', name = 'upconv6_2')(upconv6_1)
  concat_6 = concatenate([upconv6_2, conv2_2], axis = 3, name = 'concat_6') # Split Connections

  conv6_1 = Conv2D(filters = 64, kernel_size = (3, 3), padding = 'same', activation = 'relu', name = 'conv6_1')(concat_6)
  conv6_2 = Conv2D(filters = 64, kernel_size = (3, 3), padding = 'same', activation = 'relu', name = 'conv6_2')(conv6_1)

  upconv7_1 = UpSampling2D(name = 'upconv7_1')(conv6_2)
  upconv7_2 = Conv2D(filters = 32, kernel_size = (2, 2), activation = 'relu', padding = 'same', name = 'upconv7_2')(upconv7_1)
  concat_7 = concatenate([upconv7_2, conv1_2], axis = 3, name = 'concat_7') # Split Connections

  conv7_1 = Conv2D(filters = 32, kernel_size = (3, 3), padding = 'same', activation = 'relu', name = 'conv7_1')(concat_7)
  conv7_2 = Conv2D(filters = 32, kernel_size = (3, 3), padding = 'same', activation = 'relu', name = 'conv7_2')(conv7_1)

  conv8 = Conv2D(filters = num_of_classes, kernel_size = (1, 1), activation = 'softmax', name = 'conv8')(conv7_2)

  model = Model(inputs = input_image, outputs = conv8, name = 'model')

  return model

In [ ]:
unet_model = mini_u_net(image_shape = [256, 256, 3], num_of_classes = 6)

### **Описание и визуализация архитектуры созданной модели**

In [ ]:
unet_model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 256, 256, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv1_1 (Conv2D)               (None, 256, 256, 32  896         ['input_1[0][0]']                
                                )                                                                 
                                                                                                  
 conv1_2 (Conv2D)               (None, 256, 256, 32  9248        ['conv1_1[0][0]']                
                                )                                                             

In [ ]:
unet_model.get_config()

Для визуализации архитектуры модели используется следующий код:

```
plot_model(
unet_model, to_file='model.png', show_shapes=False, show_dtype=False,
show_layer_names=True, rankdir='TB', expand_nested=False, dpi=96
)
```



### **2.1. Создайте модель со следующей архитектурой:**

In [ ]:
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, Dropout, concatenate
from tensorflow.keras.models import Model

def model_2_builder(image_shape=(256, 256, 3), num_classes=5):
    inputs = Input(shape=image_shape)

    c1 = Conv2D(32, (3, 3), activation="relu", padding="same")(inputs)
    d1 = Dropout(0.1)(c1)
    c1 = Conv2D(32, (3, 3), activation="relu", padding="same")(d1)
    p1 = MaxPooling2D((2, 2))(c1)

    c2 = Conv2D(64, (3, 3), activation="relu", padding="same")(p1)
    d2 = Dropout(0.1)(c2)
    c2 = Conv2D(64, (3, 3), activation="relu", padding="same")(d2)
    p2 = MaxPooling2D((2, 2))(c2)

    c3 = Conv2D(128, (3, 3), activation="relu", padding="same")(p2)
    d3 = Dropout(0.2)(c3)
    c3 = Conv2D(128, (3, 3), activation="relu", padding="same")(d3)
    p3 = MaxPooling2D((2, 2))(c3)

    c4 = Conv2D(256, (3, 3), activation="relu", padding="same")(p3)
    d4 = Dropout(0.2)(c4)
    c4 = Conv2D(256, (3, 3), activation="relu", padding="same")(d4)
    p4 = MaxPooling2D((2, 2))(c4)

    c5 = Conv2D(512, (3, 3), activation="relu", padding="same")(p4)
    d5 = Dropout(0.3)(c5)
    c5 = Conv2D(512, (3, 3), activation="relu", padding="same")(d5)

    u6 = UpSampling2D((2, 2))(c5)
    u6 = Conv2D(256, (2, 2), activation="relu", padding="same")(u6)
    u6 = concatenate([u6, c4])
    c6 = Conv2D(256, (3, 3), activation="relu", padding="same")(u6)
    d6 = Dropout(0.2)(c6)
    c6 = Conv2D(256, (3, 3), activation="relu", padding="same")(d6)

    u7 = UpSampling2D((2, 2))(c6)
    u7 = Conv2D(128, (2, 2), activation="relu", padding="same")(u7)
    u7 = concatenate([u7, c3])
    c7 = Conv2D(128, (3, 3), activation="relu", padding="same")(u7)
    d7 = Dropout(0.2)(c7)
    c7 = Conv2D(128, (3, 3), activation="relu", padding="same")(d7)

    u8 = UpSampling2D((2, 2))(c7)
    u8 = Conv2D(64, (2, 2), activation="relu", padding="same")(u8)
    u8 = concatenate([u8, c2])
    c8 = Conv2D(64, (3, 3), activation="relu", padding="same")(u8)
    d8 = Dropout(0.1)(c8)
    c8 = Conv2D(64, (3, 3), activation="relu", padding="same")(d8)

    u9 = UpSampling2D((2, 2))(c8)
    u9 = Conv2D(32, (2, 2), activation="relu", padding="same")(u9)
    u9 = concatenate([u9, c1])
    c9 = Conv2D(32, (3, 3), activation="relu", padding="same")(u9)
    d9 = Dropout(0.1)(c9)
    c9 = Conv2D(32, (3, 3), activation="relu", padding="same")(d9)

    outputs = Conv2D(num_classes, (1, 1), activation="softmax")(c9)

    return Model(inputs=inputs, outputs=outputs, name="model_2")

try:
    model_2 = model_2_builder(image_shape=(256, 256, 3), num_classes=NUM_CLASSES)
    model_2.summary()
except Exception as e:
    print(e)

In [ ]:
model_2.summary()

In [ ]:
model_2.get_config()

In [ ]:
plot_model(
model_2, to_file='model.png', show_shapes=False, show_dtype=False,
show_layer_names=True, rankdir='TB', expand_nested=False, dpi=96
)

### **2.2. Обучите созданную модель для решения задачи семантической сегментации. ПОЭКСПЕРЕМЕНТИРУЙТЕ С [РАЗНЫМИ ФУНКЦИЯМИ ПОТЕРЬ](https://segmentation-models.readthedocs.io/en/latest/api.html#losses) И/ИЛИ ИХ КОМБИНАЦИЯМИ. Попробуйте минимум 3 разные функции потерь**

In [ ]:
# Ваш код

#### **2.2.1. Загрузите три снимка, а также маски, соответствующие этим снимкам**

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Берем 3 полноразмерных снимка из validation-набора
sample_count = 3
sample_image_names = sorted(os.listdir(val_img_dir))[:sample_count]
sample_mask_names = sorted(os.listdir(val_mask_dir))[:sample_count]

sample_images = []
sample_masks = []
sample_pairs = []

for img_name, mask_name in zip(sample_image_names, sample_mask_names):
    img_path = os.path.join(val_img_dir, img_name)
    mask_path = os.path.join(val_mask_dir, mask_name)

    image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    sample_images.append(image)
    sample_masks.append(mask)
    sample_pairs.append((img_name, mask_name))

print("Загружены пары:")
for i, (img_name, mask_name) in enumerate(sample_pairs, 1):
    print(f"{i}. {img_name} | {mask_name} | image shape={sample_images[i-1].shape} | mask shape={sample_masks[i-1].shape}")

# Быстрая проверка
plt.figure(figsize=(15, 10))
for i in range(sample_count):
    plt.subplot(sample_count, 2, 2*i + 1)
    plt.imshow(sample_images[i])
    plt.title(f"Image: {sample_pairs[i][0]}")
    plt.axis("off")

    plt.subplot(sample_count, 2, 2*i + 2)
    plt.imshow(sample_masks[i], cmap="gray")
    plt.title(f"Mask: {sample_pairs[i][1]} | classes={np.unique(sample_masks[i])}")
    plt.axis("off")

plt.tight_layout()
plt.show()

#### **2.2.2. Используя обученную модель, обработайте снимки по технологии, рассмотренной на практическом занятии (делим снимок на части, затем каждую часть обрабатываем нейросетью, а после соединяем все части для получения полноразмерной маски исходного снимка)**

In [ ]:
best_loss = sm.losses.categorical_focal_jaccard_loss
metrics = ["accuracy", sm.metrics.iou_score]

model_2.compile(
    optimizer="Adam",
    loss=best_loss,
    metrics=metrics
)

# 3) заново создаем генераторы, чтобы не было проблем после предыдущих fit
train_gen_2 = multi_class_generator(train_generator, n_classes)
val_gen_2 = multi_class_generator(val_generator, n_classes)

# 4) шаги обучения
steps_per_epoch = math.ceil(969 / BATCH_SIZE)
validation_steps = math.ceil(243 / BATCH_SIZE)

# 5) callbacks
callbacks = [
    keras.callbacks.ModelCheckpoint(
        "best_model_2.keras",
        monitor="val_iou_score",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_iou_score",
        mode="max",
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
]

# 6) обучение
history_model_2 = model_2.fit(
    train_gen_2,
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS,
    validation_data=val_gen_2,
    validation_steps=validation_steps,
    callbacks=callbacks,
    verbose=1
)

#### **2.2.3. Загрузите модель из предыдущего задания (Задания №1). Сравните её точность с моделью, обученной в этом задании:**

In [ ]:
import pandas as pd
from tensorflow.keras.metrics import MeanIoU

# Лучшая модель из задания 1
best_model_task1 = linknet_model

def predict_full_image_by_tiles(model, image, tile_size=256, num_classes=5):
    h, w = image.shape[:2]
    pred_mask = np.zeros((h, w), dtype=np.uint8)

    for y in range(0, h, tile_size):
        for x in range(0, w, tile_size):
            tile = image[y:y+tile_size, x:x+tile_size]
            tile_h, tile_w = tile.shape[:2]

            padded_tile = np.zeros((tile_size, tile_size, 3), dtype=np.uint8)
            padded_tile[:tile_h, :tile_w] = tile

            tile_input = padded_tile.astype(np.float32) / 255.0
            tile_input = np.expand_dims(tile_input, axis=0)

            tile_pred = model.predict(tile_input, verbose=0)
            tile_pred = np.argmax(tile_pred[0], axis=-1).astype(np.uint8)

            pred_mask[y:y+tile_h, x:x+tile_w] = tile_pred[:tile_h, :tile_w]

    return pred_mask

def calc_pixel_accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def calc_mean_iou(y_true, y_pred, num_classes):
    metric = MeanIoU(num_classes=num_classes)
    metric.update_state(y_true.reshape(-1), y_pred.reshape(-1))
    return float(metric.result().numpy())

comparison_rows = []

pred_masks_model_2 = []
pred_masks_best_model = []

for i, (image, true_mask) in enumerate(zip(sample_images, sample_masks), 1):
    pred_2 = predict_full_image_by_tiles(model_2, image, tile_size=256, num_classes=n_classes)
    pred_best = predict_full_image_by_tiles(best_model_task1, image, tile_size=256, num_classes=n_classes)

    pred_masks_model_2.append(pred_2)
    pred_masks_best_model.append(pred_best)

    acc_2 = calc_pixel_accuracy(true_mask, pred_2)
    iou_2 = calc_mean_iou(true_mask, pred_2, n_classes)

    acc_best = calc_pixel_accuracy(true_mask, pred_best)
    iou_best = calc_mean_iou(true_mask, pred_best, n_classes)

    comparison_rows.append({
        "image_id": i,
        "model_2_acc": round(acc_2, 4),
        "model_2_mIoU": round(iou_2, 4),
        "best_model_acc": round(acc_best, 4),
        "best_model_mIoU": round(iou_best, 4)
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

print("Средние метрики:")
print(f"model_2      -> ACC: {comparison_df['model_2_acc'].mean():.4f}, mIoU: {comparison_df['model_2_mIoU'].mean():.4f}")
print(f"best_model_1 -> ACC: {comparison_df['best_model_acc'].mean():.4f}, mIoU: {comparison_df['best_model_mIoU'].mean():.4f}")

if comparison_df["model_2_mIoU"].mean() > comparison_df["best_model_mIoU"].mean():
    print("Лучшая модель на 3 полноразмерных снимках: model_2")
else:
    print("Лучшая модель на 3 полноразмерных снимках: linknet_model")

In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
from tensorflow.keras.metrics import MeanIoU
import segmentation_models as sm

# Загружаем лучшую модель из задания 1
best_model_task1 = load_model(
    "linknet_model.keras",
    custom_objects={
        "categorical_focal_jaccard_loss": sm.losses.categorical_focal_jaccard_loss,
        "iou_score": sm.metrics.iou_score
    },
    compile=False
)

best_model_task1.compile(
    optimizer="Adam",
    loss=sm.losses.categorical_focal_jaccard_loss,
    metrics=["accuracy", sm.metrics.iou_score]
)

def calc_pixel_accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def calc_mean_iou(y_true, y_pred, num_classes):
    metric = MeanIoU(num_classes=num_classes)
    metric.update_state(y_true.reshape(-1), y_pred.reshape(-1))
    return float(metric.result().numpy())

# 1) Сравнение на всей валидации по обычному evaluate
y_val_cat = tf.keras.utils.to_categorical(y_val, num_classes=n_classes).astype(np.float32)

loss_model_2, acc_model_2, iou_model_2 = model_2.evaluate(X_val_norm, y_val_cat, verbose=1)
loss_best, acc_best, iou_best = best_model_task1.evaluate(X_val_norm, y_val_cat, verbose=1)

# 2) Дополнительно считаем mean IoU по argmax-предсказаниям
pred_model_2 = np.argmax(model_2.predict(X_val_norm, verbose=1), axis=-1)
pred_best = np.argmax(best_model_task1.predict(X_val_norm, verbose=1), axis=-1)

pixel_acc_model_2 = calc_pixel_accuracy(y_val, pred_model_2)
pixel_acc_best = calc_pixel_accuracy(y_val, pred_best)

miou_model_2 = calc_mean_iou(y_val, pred_model_2, n_classes)
miou_best = calc_mean_iou(y_val, pred_best, n_classes)

comparison_df = pd.DataFrame([
    {
        "model": "model_2",
        "eval_loss": round(loss_model_2, 4),
        "eval_accuracy": round(acc_model_2, 4),
        "eval_iou_score": round(iou_model_2, 4),
        "pixel_accuracy": round(pixel_acc_model_2, 4),
        "mean_iou_argmax": round(miou_model_2, 4)
    },
    {
        "model": "linknet_model",
        "eval_loss": round(loss_best, 4),
        "eval_accuracy": round(acc_best, 4),
        "eval_iou_score": round(iou_best, 4),
        "pixel_accuracy": round(pixel_acc_best, 4),
        "mean_iou_argmax": round(miou_best, 4)
    }
])

display(comparison_df)

if miou_model_2 > miou_best:
    better_model_name = "model_2"
else:
    better_model_name = "linknet_model"

print(f"Лучшая модель по mean_iou_argmax на всей валидационной выборке: {better_model_name}")

#### **2.2.4. Отобразите предсказанную маску для каждого из трех снимков, загруженных ранее, в следующем формате: (исходный снимок, эталонная маска, предсказанная маска по модели, обученной в этом задании, предсказанная маска по модели, обученной в первом задании.**

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

sample_count = 3
background_class = 2  # по твоим примерам основной фон = класс 2

all_image_names = sorted(os.listdir(val_img_dir))
all_mask_names = sorted(os.listdir(val_mask_dir))

candidates = []

for img_name, mask_name in zip(all_image_names, all_mask_names):
    img_path = os.path.join(val_img_dir, img_name)
    mask_path = os.path.join(val_mask_dir, mask_name)

    image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    unique_classes = np.unique(mask)
    non_background_ratio = np.mean(mask != background_class)
    num_classes = len(unique_classes)

    # чем больше классов и чем больше не-фона, тем интереснее пример
    score = num_classes * 10 + non_background_ratio

    candidates.append({
        "img_name": img_name,
        "mask_name": mask_name,
        "image": image,
        "mask": mask,
        "unique_classes": unique_classes,
        "non_background_ratio": non_background_ratio,
        "score": score
    })

# сортируем по "интересности"
candidates = sorted(
    candidates,
    key=lambda x: (len(x["unique_classes"]), x["non_background_ratio"]),
    reverse=True
)

selected = candidates[:sample_count]

sample_images = [x["image"] for x in selected]
sample_masks = [x["mask"] for x in selected]
sample_pairs = [(x["img_name"], x["mask_name"]) for x in selected]

print("Выбраны более содержательные снимки:")
for i, x in enumerate(selected, 1):
    print(
        f"{i}. {x['img_name']} | {x['mask_name']} | "
        f"classes={x['unique_classes']} | "
        f"non_background_ratio={x['non_background_ratio']:.4f}"
    )

plt.figure(figsize=(15, 10))
for i in range(sample_count):
    plt.subplot(sample_count, 2, 2 * i + 1)
    plt.imshow(sample_images[i])
    plt.title(f"Image: {sample_pairs[i][0]}")
    plt.axis("off")

    plt.subplot(sample_count, 2, 2 * i + 2)
    plt.imshow(sample_masks[i], cmap="gray")
    plt.title(f"Mask: {sample_pairs[i][1]} | classes={np.unique(sample_masks[i])}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Предсказания для 3 ранее загруженных снимков
pred_masks_model_2 = []
pred_masks_best_model = []

for image in sample_images:
    pred_2 = predict_full_image_by_tiles(model_2, image, tile_size=256, num_classes=n_classes)
    pred_best = predict_full_image_by_tiles(best_model_task1, image, tile_size=256, num_classes=n_classes)

    pred_masks_model_2.append(pred_2)
    pred_masks_best_model.append(pred_best)

# Визуализация:
# (исходный снимок, эталонная маска, pred model_2, pred model from task 1)
num_samples = len(sample_images)

plt.figure(figsize=(20, 5 * num_samples))

for i in range(num_samples):
    # 1. Исходный снимок
    plt.subplot(num_samples, 4, i * 4 + 1)
    plt.imshow(sample_images[i])
    plt.title(f"Снимок {i+1}")
    plt.axis("off")

    # 2. Эталонная маска
    plt.subplot(num_samples, 4, i * 4 + 2)
    plt.imshow(sample_masks[i], cmap="gray", vmin=0, vmax=n_classes-1)
    plt.title("Эталонная маска")
    plt.axis("off")

    # 3. Предсказание model_2
    plt.subplot(num_samples, 4, i * 4 + 3)
    plt.imshow(pred_masks_model_2[i], cmap="gray", vmin=0, vmax=n_classes-1)
    plt.title("Маска: model_2")
    plt.axis("off")

    # 4. Предсказание лучшей модели из задания 1
    plt.subplot(num_samples, 4, i * 4 + 4)
    plt.imshow(pred_masks_best_model[i], cmap="gray", vmin=0, vmax=n_classes-1)
    plt.title("Маска: модель из задания 1")
    plt.axis("off")

plt.tight_layout()
plt.show()

---


## **Задание №3. Реализация готового решения для полноразмерной семантической сегментации спутникового снимка**

Реализуйте итоговый скрипт, который принимает на вход полноразмерное изображение и формирует для него предсказанную маску, используя лучшую модель из предыдущих этапов.


**Требования:**
1. Алгоритм: Реализовать инференс на основе наиболее точной модели (из Задания №1 или №2) с автоматическим сохранением результата в корневой каталог.
2. Интерфейс: Создать веб-приложение на базе Gradio/Streamlit (на крайний случай [таким образом](https://colab.research.google.com/drive/1AOAlbgdnPcr94qJFyMumVpnVezuDnocy?usp=sharing#scrollTo=0JY635HtBYA2)), обеспечивающее загрузку исходного растра и визуализацию полученной маски.
3. Функционал: Предусмотреть корректную обработку полноразмерных изображений, с использованием плавного смешивания.

In [ ]:
import os
os.environ["SM_FRAMEWORK"] = "tf.keras"

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.metrics import MeanIoU
import segmentation_models as sm

# ====== Константы ======
MODEL_PATH = "linknet_model.keras"   # лучшая модель из задания №1
PATCH_SIZE = 256
N_CLASSES = 5

# Палитра классов из теории (5 классов)
CLASS_COLORS = np.array([
    [60, 16, 152],    # 0 - building
    [132, 41, 246],   # 1 - land
    [110, 193, 228],  # 2 - road
    [254, 221, 58],   # 3 - vegetation
    [226, 169, 41],   # 4 - water
], dtype=np.uint8)

# ====== Загрузка модели ======
best_model = load_model(
    MODEL_PATH,
    custom_objects={
        "categorical_focal_jaccard_loss": sm.losses.categorical_focal_jaccard_loss,
        "iou_score": sm.metrics.iou_score
    },
    compile=False
)

best_model.compile(
    optimizer="Adam",
    loss=sm.losses.categorical_focal_jaccard_loss,
    metrics=["accuracy", sm.metrics.iou_score]
)

# ====== Вспомогательные функции ======
def label_to_rgb(mask_2d: np.ndarray) -> np.ndarray:
    mask_2d = mask_2d.astype(np.uint8)
    return CLASS_COLORS[mask_2d]

def predict_full_image_by_tiles(model, image_rgb: np.ndarray, tile_size: int = 256) -> np.ndarray:
    h, w = image_rgb.shape[:2]

    pad_h = (tile_size - h % tile_size) % tile_size
    pad_w = (tile_size - w % tile_size) % tile_size

    padded = cv2.copyMakeBorder(
        image_rgb, 0, pad_h, 0, pad_w,
        borderType=cv2.BORDER_REFLECT_101
    )

    ph, pw = padded.shape[:2]
    pred_mask = np.zeros((ph, pw), dtype=np.uint8)

    for y in range(0, ph, tile_size):
        for x in range(0, pw, tile_size):
            tile = padded[y:y+tile_size, x:x+tile_size]
            tile_input = tile.astype(np.float32) / 255.0
            tile_input = np.expand_dims(tile_input, axis=0)

            pred = model.predict(tile_input, verbose=0)
            pred = np.argmax(pred[0], axis=-1).astype(np.uint8)

            pred_mask[y:y+tile_size, x:x+tile_size] = pred

    return pred_mask[:h, :w]

def calc_pixel_accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(y_true == y_pred))

def calc_mean_iou(y_true: np.ndarray, y_pred: np.ndarray, num_classes: int = 5) -> float:
    metric = MeanIoU(num_classes=num_classes)
    metric.update_state(y_true.reshape(-1), y_pred.reshape(-1))
    return float(metric.result().numpy())

def run_full_segmentation(image_path: str, mask_path: str = None, save_name: str = "predicted_mask.png"):
    image_bgr = cv2.imread(image_path)
    if image_bgr is None:
        raise FileNotFoundError(f"Не удалось загрузить изображение: {image_path}")

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    pred_mask = predict_full_image_by_tiles(best_model, image_rgb, tile_size=PATCH_SIZE)
    pred_mask_rgb = label_to_rgb(pred_mask)

    cv2.imwrite(save_name, cv2.cvtColor(pred_mask_rgb, cv2.COLOR_RGB2BGR))
    print(f"Предсказанная маска сохранена: {save_name}")

    plt.figure(figsize=(18, 6))

    plt.subplot(1, 3, 1)
    plt.imshow(image_rgb)
    plt.title("Исходный снимок")
    plt.axis("off")

    if mask_path is not None:
        true_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if true_mask is None:
            raise FileNotFoundError(f"Не удалось загрузить маску: {mask_path}")

        plt.subplot(1, 3, 2)
        plt.imshow(label_to_rgb(true_mask))
        plt.title("Эталонная маска")
        plt.axis("off")

        plt.subplot(1, 3, 3)
        plt.imshow(pred_mask_rgb)
        plt.title("Предсказанная маска")
        plt.axis("off")

        plt.tight_layout()
        plt.show()

        pixel_acc = calc_pixel_accuracy(true_mask, pred_mask)
        mean_iou = calc_mean_iou(true_mask, pred_mask, num_classes=N_CLASSES)

        print(f"Pixel Accuracy: {pixel_acc:.4f}")
        print(f"Mean IoU: {mean_iou:.4f}")

        return pred_mask, pred_mask_rgb, true_mask
    else:
        plt.subplot(1, 3, 2)
        plt.imshow(pred_mask_rgb)
        plt.title("Предсказанная маска")
        plt.axis("off")

        plt.tight_layout()
        plt.show()

        return pred_mask, pred_mask_rgb

In [ ]:

image_path = "./landcover.ai.v1/images/" + sorted(os.listdir("./landcover.ai.v1/images"))[0]
mask_path  = "./landcover.ai.v1/masks/"  + sorted(os.listdir("./landcover.ai.v1/masks"))[0]

pred_mask, pred_mask_rgb, true_mask = run_full_segmentation(
    image_path=image_path,
    mask_path=mask_path,
    save_name="predicted_mask_from_full_image.png"
)

# Дополнительная визуализация ошибки
diff_map = (pred_mask != true_mask).astype(np.uint8)

plt.figure(figsize=(20, 5))

plt.subplot(1, 4, 1)
img_rgb = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
plt.imshow(img_rgb)
plt.title("Исходный снимок")
plt.axis("off")

plt.subplot(1, 4, 2)
plt.imshow(label_to_rgb(true_mask))
plt.title("Эталонная маска")
plt.axis("off")

plt.subplot(1, 4, 3)
plt.imshow(pred_mask_rgb)
plt.title("Предсказанная маска")
plt.axis("off")

plt.subplot(1, 4, 4)
plt.imshow(diff_map, cmap="Reds")
plt.title("Карта ошибок")
plt.axis("off")

plt.tight_layout()
plt.show()

print("Уникальные классы в эталоне:", np.unique(true_mask))
print("Уникальные классы в предсказании:", np.unique(pred_mask))
print(f"Pixel Accuracy: {calc_pixel_accuracy(true_mask, pred_mask):.4f}")
print(f"Mean IoU: {calc_mean_iou(true_mask, pred_mask, num_classes=N_CLASSES):.4f}")

In [ ]:
import gradio as gr
from PIL import Image
import os

# ====== Подготовка examples ======
test_images_dir = "./landcover.ai.v1/images"
example_images = sorted(os.listdir(test_images_dir))[:5]  # берем 5 примеров

examples = [[os.path.join(test_images_dir, img)] for img in example_images]

# ====== Gradio функция ======
def predict_mask_gradio(input_image):
    if input_image is None:
        return None

    if isinstance(input_image, Image.Image):
        image_rgb = np.array(input_image.convert("RGB"))
    else:
        image_rgb = np.array(input_image)

    pred_mask = predict_full_image_by_tiles(best_model, image_rgb, tile_size=PATCH_SIZE)
    pred_mask_rgb = label_to_rgb(pred_mask)

    save_path = "gradio_predicted_mask.png"
    cv2.imwrite(save_path, cv2.cvtColor(pred_mask_rgb, cv2.COLOR_RGB2BGR))

    return pred_mask_rgb

# ====== Интерфейс ======
demo = gr.Interface(
    fn=predict_mask_gradio,
    inputs=gr.Image(type="pil", label="Исходный растр"),
    outputs=gr.Image(type="numpy", label="Сгенерированная маска"),
    examples=examples,
    title="Семантическая сегментация спутникового снимка",
    description="Примеры загружены из тестовой выборки landcover.ai.v1"
)

demo.launch(share=False, debug=True)

---


## **Задание №4. Обучение модели семантической сегментации с последующим [квантованием](https://colab.research.google.com/drive/1hyK_nN_o1FhySZrxjII0Jm3g-6HvsdZv#scrollTo=wWSrNOR4eCG3)**

---

**БАКАЛАВРАМ:**

Обучите модель сегментации на датасете из часто встречающихся в реальной жизни объектах (элементы интерьера, мебель, техника, улица и т.д.).



Примеры датасетов:

1. [Cityscapes](https://www.kaggle.com/datasets/shuvoalok/cityscapes)
2. [Ade20k](https://www.kaggle.com/datasets/shubhjyot/ade20k)
3. [NYU Depth V2](https://www.kaggle.com/datasets/soumikrakshit/nyu-depth-v2)

---

**МАГИСТРАНТАМ:**

Обучите модель сегментации на датасете, который представлен ниже


unet_model.compile(
    optimizer=Adam(learning_rate=0.0001),
    # Для многоклассовой сегментации softmax используется CategoricalCELoss()
    loss=sm.losses.CategoricalCELoss(),
    metrics=[sm.metrics.IOUScore(threshold=0.5)]
)

# Требуется 3 разные ф-ции потерь и обучение.
# Для экономии времени запишем циклом с разными Loss
losses_to_test = {
    'CrossEntropy': sm.losses.CategoricalCELoss(),
    'Dice': sm.losses.DiceLoss(),
    'Focal': sm.losses.CategoricalFocalLoss(),
    'Jaccard': sm.losses.JaccardLoss(),
}

models_history = {}

for loss_name, loss_fn in losses_to_test.items():
    print(f"\n--- Обучение модели с функцией потерь: {loss_name} ---")
    
    # 1. Пересоздаем модель каждый раз, чтобы веса были нулевыми (с нуля)
    current_model = mini_u_net(image_shape=[256, 256, 3], num_of_classes=NUM_CLASSES)
    
    # 2. Компилируем
    current_model.compile(
        optimizer=Adam(learning_rate=0.001), 
        loss=loss_fn, 
        metrics=[sm.metrics.IOUScore(threshold=0.5)]
    )
    
    # 3. Уменьшим кол-во эпох для 4 экспериментов, иначе слишком долго будет считаться
    hist = current_model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=10, 
        verbose=1
    )
    models_history[loss_name] = {'history': hist, 'model': current_model}
    
# Выбираем лучшую модель из 4-х
best_loss_name = max(models_history, key=lambda k: models_history[k]['history'].history['val_iou_score'][-1])
print(f"\nЛучшая функция потерь на валидации: {best_loss_name}")
best_custom_model = models_history[best_loss_name]['model']

In [ ]:
# Извлекаем три изображения и их маски
test_images_paths = all_image_paths[:3]
test_masks_paths = all_mask_paths[:3]

test_image_arrays = []
test_mask_arrays = []

for idx in range(3):
    img = np.array(Image.open(test_images_paths[idx]))
    mask = np.array(Image.open(test_masks_paths[idx]))
    test_image_arrays.append(img)
    test_mask_arrays.append(mask)
    
print("Загружено 3 полноразмерных снимка и 3 маски.")

Проведите эксперименты с квантованием модели в разные форматы и определите оптимальную по метрикам и информационному объему (весу в Мб).



In [ ]:
fast_predicts_task_2 = []
smooth_predicts_task_2 = []

for i in range(3):
    img = test_image_arrays[i]
    print(f"\nОбработка изображения {i+1} моделью из Задания 2 (Custom U-Net)")
    
    pred_fast = fast_predict(img, best_custom_model, patch_size=256)
    pred_smooth = smooth_predict(img, best_custom_model, patch_size=256, stride=128)
    
    fast_predicts_task_2.append(pred_fast)
    smooth_predicts_task_2.append(pred_smooth)

Визуализируйте графики с метриками и таблицу с информационным объемом моделей.

In [ ]:
# Сравниваем Custom-модель с моделью ResNet34-UNet (которую мы сохраняли в 'best_unet_model.h5')
resnet_unet_model = load_model('best_unet_model.h5', compile=False)

fast_predicts_task_1 = []
print("Идет вычисление масок с использованием модели из Задания 1")
for i in range(3):
    img = test_image_arrays[i]
    pred_fast = fast_predict(img, resnet_unet_model, patch_size=256)
    fast_predicts_task_1.append(pred_fast)
    
# Сравним средний IoU на 3 снимках
iou_t1 = []
iou_t2 = []

for i in range(3):
    mask = test_mask_arrays[i]
    
    # Приводим к 1 классу (индексы) 
    if len(mask.shape) == 3:
        colors = [CLASS_COLORS[c] for c in range(NUM_CLASSES)]
        mask_idx = np.zeros(mask.shape[:2], dtype=np.uint8)
        for idx_col, c in enumerate(colors):
            mask_idx[(mask == c).all(axis=-1)] = idx_col
        mask = mask_idx
        
    k1 = cohen_kappa_score(mask.flatten(), fast_predicts_task_1[i].flatten())
    k2 = cohen_kappa_score(mask.flatten(), fast_predicts_task_2[i].flatten())
    
    iou_t1.append(k1)
    iou_t2.append(k2)
    
print(f"Средний показатель Kappa для ResNet34 UNet: {np.mean(iou_t1):.4f}")
print(f"Средний показатель Kappa для нашей собранной сети: {np.mean(iou_t2):.4f}")

---

**БАКАЛАВРАМ:**

- После выбора оптимальной по объему и качеству модели, необходимо интегрировать её [в мобильное приложение](https://colab.research.google.com/drive/102Oup8wfyYQDhw_3V-VsuOwh_ZuJWTgU?usp=sharing#scrollTo=132b0766).

- Если в мобильном приложении возникают ошибки при попытке сделать прогноз, то замените каждый слой Conv2DTranspose в модели на связку UpSampling2D + Conv2D. Это поможет избежать проблем с неподдерживаемыми операциями (ops) в TFLite

In [ ]:
import tensorflow as tf
import os

# 1. Сохранение лучших моделей в TFLite без квантования (FP32)
def convert_to_tflite(model, filename):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_model = converter.convert()
    with open(filename, 'wb') as f:
        f.write(tflite_model)
    print(f"Модель сохранена: {filename} (Размер: {os.path.getsize(filename) / (1024 * 1024):.2f} МБ)")

# 2. Квантование (Dynamic Range - FP16 weights / INT8 activations)
def convert_to_tflite_quantized(model, filename):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_quant_model = converter.convert()
    with open(filename, 'wb') as f:
        f.write(tflite_quant_model)
    print(f"Квантованная модель сохранена: {filename} (Размер: {os.path.getsize(filename) / (1024 * 1024):.2f} МБ)")

print("=== Конвертация кастомной модели (Задание 2) ===")
convert_to_tflite(best_custom_model, 'custom_unet.tflite')
convert_to_tflite_quantized(best_custom_model, 'custom_unet_quantized.tflite')

print("\n=== Конвертация базовой модели (Задание 1) ===")
# Если нужно конвертировать и базовую модель для сравнения
convert_to_tflite(resnet_unet_model, 'resnet_unet.tflite')
convert_to_tflite_quantized(resnet_unet_model, 'resnet_unet_quantized.tflite')